In [ ]:
import pandas as pd
import numpy as np
import os
import requests
import json
import datetime
import time

In [ ]:
BUFFER = 1

base_url = "https://www.kaggle.com/requests/EpisodeService/"
get_url = base_url + "GetEpisodeReplay"
list_url = base_url + "ListEpisodes"

In [ ]:
# inital team list

r = requests.post(list_url, json = {"submissionId":  18703109}) # ID is sample value
rj = r.json()

teams_df = pd.DataFrame(rj['result']['teams'])

In [ ]:
teams_df.sort_values('publicLeaderboardRank', inplace = True)
teams_df.head(15)

## Get Episodes of top teams

In [ ]:
def saveEpisode(sub_id, epid, rj):
    # request
    re = requests.post(get_url, json = {"EpisodeId": int(epid)})
        
    # save replay
    with open('{}_{}.json'.format(sub_id, epid), 'w') as f:
        f.write(re.json()['result']['replay'])

    # save episode info
    with open('{}_{}_info.json'.format(sub_id, epid), 'w') as f:
        json.dump([r for r in rj['result']['episodes'] if r['id']==epid][0], f)

In [ ]:
for ind, sub in enumerate(teams_df['publicLeaderboardSubmissionId'].tolist()[0:15]):
    print("Position {}".format(str(ind)))
    start_time = datetime.datetime.now()
    r = BUFFER;
    result = requests.post(list_url, json = {"submissionId":  int(sub)})
    team_json = result.json()
    team_df = pd.DataFrame(team_json['result']['episodes'])
    print('{} games for {}'.format(len(team_df), sub))

    for i in range(len(team_df)):
        epid = team_df.id.iloc[i]

        saveEpisode(sub, epid, team_json); r+=1;
        try:
            size = os.path.getsize('{}_{}.json'.format(sub, epid)) / 1e6
            print('Saved Episode #{} @ {:.1f}MB'.format(epid, size))
        except:
            print('file {}_{}.json did not seem to save'.format(sub, epid))    
        if r > (datetime.datetime.now() - start_time).seconds:
            time.sleep( r - (datetime.datetime.now() - start_time).seconds)

In [ ]:
127+54+34+61+35+94+44+34+116+60+61+47+43+127+297

In [ ]:
!rm *_info.json

In [ ]:
!ls /kaggle/working/ | wc -l

In [ ]:
!cd /kaggle/working/ && tar -czvf /kaggle/working/top15LB2.tar.gz *.json

In [ ]:
!rm *.json

In [ ]:
!ls /kaggle/working/ | wc -l